# Phase 3, experiment 8: tuning LightGBM

**Why this experiment:** LightGBM with default settings is still the best model: it beat every feature
change and the three-model ensemble on both the fold mean and the public leaderboard. Its training AUC
is 1.0, so it memorises the training rows, and there may be settings that generalise better.

**Decision:** use LightGBM alone going forward (the ensemble with XGBoost and Random
Forest lowered the fold mean by 0.0085 and the public leaderboard by 0.0058).

**How the search works: one setting at a time.** Starting from the defaults, each setting is changed
on its own while everything else stays at the default. This is less thorough than searching all
combinations, but it is quick, and it shows clearly what each setting does.

| Setting | What it controls | Values tried (default in bold) |
|---|---|---|
| `num_leaves` | maximum leaves per tree (tree complexity) | 7, 15, **31**, 63 |
| `min_child_samples` | minimum training rows in a leaf | 5, **20**, 50, 100 |
| `colsample_bytree` | share of features each tree may use | 0.3, 0.5, 0.8, **1.0** |
| `subsample` | share of rows each tree is trained on | 0.5, 0.7, **1.0** |
| `reg_lambda` | penalty on large leaf values | **0**, 1, 5, 10 |
| trees and learning rate | many small steps against few large ones | **100 at 0.1**, 300 at 0.05, 500 at 0.03 |

**How a setting is judged:** Notebook 07's folds are used, and the fold mean has matched the direction
of the public leaderboard three times in a row. Fold 2 (with step 24) is the one that shows how the
model copes with new kinds of illicit behaviour, and it is also the noisiest (similar models differ by
0.01 to 0.02 there). So a setting is only worth keeping if it raises the fold mean without lowering
fold 2, and ideally if neighbouring values point the same way, rather than a single value that happens
to score well.

Features: the 165 raw features.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

## 2. Folds and a function that scores one set of settings

In [ ]:
# Same three expanding-window folds as notebook 07
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]

def split_fold(df, fold):
    # Return the (training rows, validation rows) of df for one fold.
    tr = df[df["time_step"] <= fold["train_last"]]
    va = df[(df["time_step"] > fold["train_last"]) & (df["time_step"] <= fold["valid_last"])]
    return tr, va

def score_settings(params):
    # Train LightGBM with the given settings on every fold and return the AUC of each fold.
    # params holds only what differs from the defaults
    result = {}
    for fold in FOLDS:
        tr, va = split_fold(labeled, fold)
        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **params)
        model.fit(tr[raw_cols], tr["label"])
        va_pred = model.predict_proba(va[raw_cols])[:, 1]
        result[fold["name"]] = roc_auc_score(va["label"], va_pred)
    return result

## 3. Change one setting at a time

Each entry in `candidates` is one model to score: a label and the settings that differ from the
defaults. `subsample` only takes effect when `subsample_freq` is above 0, so that is set to 1
(draw a new sample of rows for every tree).

In [3]:
candidates = [("default", {})]

for value in [7, 15, 63]:
    candidates.append((f"num_leaves={value}", {"num_leaves": value}))
for value in [5, 50, 100]:
    candidates.append((f"min_child_samples={value}", {"min_child_samples": value}))
for value in [0.3, 0.5, 0.8]:
    candidates.append((f"colsample_bytree={value}", {"colsample_bytree": value}))
for value in [0.5, 0.7]:
    candidates.append((f"subsample={value}", {"subsample": value, "subsample_freq": 1}))
for value in [1, 5, 10]:
    candidates.append((f"reg_lambda={value}", {"reg_lambda": value}))
for n_trees, rate in [(300, 0.05), (500, 0.03)]:
    candidates.append((f"trees={n_trees}, lr={rate}", {"n_estimators": n_trees, "learning_rate": rate}))

print("Number of settings to score:", len(candidates), "(each trained on 3 folds)")

rows = []
for label, params in candidates:
    fold_auc = score_settings(params)
    rows.append({"settings": label, **fold_auc})
    print("done:", label)

tuning = pd.DataFrame(rows).set_index("settings")
tuning["mean"] = tuning[fold_names].mean(axis=1)

Number of settings to score: 17 (each trained on 3 folds)
done: default
done: num_leaves=7
done: num_leaves=15
done: num_leaves=63
done: min_child_samples=5
done: min_child_samples=50
done: min_child_samples=100
done: colsample_bytree=0.3
done: colsample_bytree=0.5
done: colsample_bytree=0.8
done: subsample=0.5
done: subsample=0.7
done: reg_lambda=1
done: reg_lambda=5
done: reg_lambda=10
done: trees=300, lr=0.05
done: trees=500, lr=0.03


## 4. Results

`change_mean` and `change_fold2` compare each setting with the default (first row). A setting is a
candidate to keep only if both are positive.

In [5]:
default_row = tuning.loc["default"]
tuning["change_mean"] = tuning["mean"] - default_row["mean"]
tuning["change_fold2"] = tuning["fold 2"] - default_row["fold 2"]
tuning.round(4)

,fold 1,fold 2,fold 3,mean,change_mean,change_fold2
settings,,,,,,
default,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000
num_leaves=7,0.9919,0.9540,0.9950,0.9803,-0.0005,-0.0032
num_leaves=15,0.9929,0.9487,0.9963,0.9793,-0.0015,-0.0085
num_leaves=63,0.9877,0.9368,0.9956,0.9734,-0.0075,-0.0204
min_child_samples=5,0.9858,0.9398,0.9958,0.9738,-0.0070,-0.0174
min_child_samples=50,0.9926,0.9403,0.9964,0.9765,-0.0044,-0.0169
min_child_samples=100,0.9937,0.9409,0.9964,0.9770,-0.0038,-0.0163
colsample_bytree=0.3,0.9937,0.9493,0.9970,0.9800,-0.0008,-0.0079
colsample_bytree=0.5,0.9932,0.9515,0.9972,0.9806,-0.0002,-0.0057


In [6]:
# Only the settings that raise the mean without lowering fold 2
keepers = tuning[(tuning["change_mean"] > 0) & (tuning["change_fold2"] >= 0)]
keepers.sort_values("change_mean", ascending=False).round(4)

,fold 1,fold 2,fold 3,mean,change_mean,change_fold2
settings,,,,,,


## Findings

- **Which settings raise the fold mean without lowering fold 2?** None. The closest is `reg_lambda=5`,
  which raises the mean by 0.0003 and lowers fold 2 by 0.0009, both far smaller than the noise on
  these folds.
- **Do neighbouring values point the same way (a trend), or is it a single value?** There is one clear
  pattern across all 16 alternatives. Fold 1 goes up for 15 of them (by up to 0.005; only
  `min_child_samples=5` goes down), fold 3 stays within about 0.0015 of the default, and fold 2 goes down
  for all 16 (by 0.001 to 0.038). So the default's fold 2 score (0.9572) is at the top of the range that
  similar settings reach there (0.919 to 0.956). This could mean the default is partly lucky on fold 2,
  or that the default settings happen to suit new kinds of illicit behaviour. This data cannot tell the
  two apart, but the public leaderboard has also favoured the default LightGBM over every change tried
  so far.
- **Which settings hurt?** The largest drops in the mean come from `subsample=0.5` (-0.0116, fold 2
  -0.0383), `num_leaves=63` (-0.0075) and `min_child_samples=5` (-0.0070), which are the settings that
  make the model either much noisier or much more complex.
- **Decision: which settings to combine and test next?** None. Keep the LightGBM defaults, because no
  setting raised the fold mean without lowering fold 2.
